# Polyphage

Multi-objective design of IsPETase variants: ProteinMPNN proposes sequences
against a fixed backbone with the catalytic triad frozen, NSGA-II searches that
space under competing objectives and returns a Pareto front.

**Runtime: T4 GPU** (Runtime > Change runtime type > T4 GPU). Free tier is enough.

Run cells in order. Cell 4 and Cell 6 are gates -- if either raises, stop and fix
it rather than continuing.

Reference structures: `6EQE` = wild-type IsPETase (the design target),
`7SH6` = FAST-PETase (benchmark only). Earlier `6EQM`/`7VVE` were wrong;
`7VVE` is not FAST-PETase.

## Cell 1 — Environment

In [ ]:
!git clone https://github.com/dauparas/ProteinMPNN.git
%cd ProteinMPNN
!pip install -q biopython pymoo matplotlib
!mkdir -p polyphage outputs inputs/pdbs inputs/design_target

import sys
sys.path.insert(0, '/content/ProteinMPNN')
print('ready')

## Cell 2 — Write the polyphage package

These cells write the project modules to disk. Nothing to upload or mount.

In [ ]:
%%writefile polyphage/__init__.py
"""Polyphage: multi-objective design of IsPETase variants."""

from .encoding import (
    AA_ALPHABET,
    CATALYTIC_TRIAD,
    Target,
    decode,
    format_mutations,
    hamming,
    load_target,
    mutations,
    wildtype_vector,
)
from .control import compare_to_null, repeat_seeds_summary, run_control
from .optimize import ObjectiveSpec, foldability_objective, novelty_objective, run
from .scoring import BatchScorer, SubprocessScorer, check_equivalence
from .stability import StabilityOracle, stability_objective
from .validate import (
    DURA_PETASE,
    FAST_PETASE,
    KNOWN_HOTSPOTS,
    log_run,
    plot_front,
    print_front,
    report_front,
    summarize,
)

__all__ = [
    "AA_ALPHABET", "CATALYTIC_TRIAD", "Target", "decode", "format_mutations",
    "hamming", "load_target", "mutations", "wildtype_vector",
    "BatchScorer", "SubprocessScorer", "check_equivalence",
    "ObjectiveSpec", "foldability_objective", "novelty_objective", "run",
    "StabilityOracle", "stability_objective",
    "compare_to_null", "repeat_seeds_summary", "run_control",
    "DURA_PETASE", "FAST_PETASE", "KNOWN_HOTSPOTS",
    "log_run", "plot_front", "print_front", "report_front", "summarize",
]

In [ ]:
%%writefile polyphage/encoding.py
"""Sequence <-> optimizer-vector encoding for a single design target.

The optimizer works on an integer vector with one entry per *mutable* position.
Everything that maps that vector back to a real amino-acid sequence, and back to
real PDB residue numbers, lives here so the rest of the pipeline never has to
guess about offsets.
"""

from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path

import numpy as np
from Bio.PDB import PDBParser

AA_ALPHABET = "ACDEFGHIKLMNPQRSTVWY"
AA_TO_IDX = {aa: i for i, aa in enumerate(AA_ALPHABET)}

# Catalytic triad of IsPETase, in PDB residue numbering.
CATALYTIC_TRIAD = (160, 206, 237)
TRIAD_EXPECTED = {160: "S", 206: "D", 237: "H"}

THREE_TO_ONE = {
    "ALA": "A", "ARG": "R", "ASN": "N", "ASP": "D", "CYS": "C",
    "GLN": "Q", "GLU": "E", "GLY": "G", "HIS": "H", "ILE": "I",
    "LEU": "L", "LYS": "K", "MET": "M", "PHE": "F", "PRO": "P",
    "SER": "S", "THR": "T", "TRP": "W", "TYR": "Y", "VAL": "V",
}


@dataclass
class Target:
    """A backbone to design against, plus the index bookkeeping it implies."""

    pdb_id: str
    chain: str
    wildtype_seq: str
    residue_numbers: list[int]  # PDB numbering, aligned 1:1 with wildtype_seq
    fixed_residue_numbers: tuple[int, ...]
    fixed_idx: list[int]        # 0-based positions into wildtype_seq
    mutable_idx: list[int]

    @property
    def n_mutable(self) -> int:
        return len(self.mutable_idx)

    def resnum(self, idx: int) -> int:
        """PDB residue number for a 0-based sequence position."""
        return self.residue_numbers[idx]


def load_target(
    pdb_path: str | Path,
    parsed_jsonl: str | Path,
    chain: str = "A",
    fixed_residue_numbers: tuple[int, ...] = CATALYTIC_TRIAD,
) -> Target:
    """Build a Target from a PDB file and ProteinMPNN's parsed .jsonl.

    Both are read from the same ATOM records, so the Biopython residue list and
    ProteinMPNN's sequence must come out the same length. That is asserted
    rather than assumed -- a mismatch means every downstream residue number is
    silently wrong.
    """
    pdb_path, parsed_jsonl = Path(pdb_path), Path(parsed_jsonl)
    pdb_id = pdb_path.stem

    with open(parsed_jsonl) as f:
        entry = next(json.loads(line) for line in f if line.strip())
    wildtype_seq = entry[f"seq_chain_{chain}"]

    structure = PDBParser(QUIET=True).get_structure(pdb_id, pdb_path)
    residues = [r for r in structure[0][chain] if r.id[0] == " "]
    residue_numbers = [r.id[1] for r in residues]

    if len(residue_numbers) != len(wildtype_seq):
        raise ValueError(
            f"{pdb_id} chain {chain}: Biopython sees {len(residue_numbers)} modeled "
            f"residues but ProteinMPNN parsed {len(wildtype_seq)}. Residue numbering "
            "cannot be trusted -- inspect the PDB for altlocs, insertion codes or "
            "HETATM residues before going further."
        )

    # Cross-check the two sources agree residue by residue, not just in length.
    bio_seq = "".join(THREE_TO_ONE.get(r.get_resname(), "X") for r in residues)
    if bio_seq != wildtype_seq:
        n_diff = sum(a != b for a, b in zip(bio_seq, wildtype_seq))
        raise ValueError(
            f"{pdb_id} chain {chain}: Biopython and ProteinMPNN sequences differ at "
            f"{n_diff} positions. Do not proceed; the index mapping is unreliable."
        )

    num_to_idx = {rn: i for i, rn in enumerate(residue_numbers)}
    missing = [rn for rn in fixed_residue_numbers if rn not in num_to_idx]
    if missing:
        raise ValueError(f"{pdb_id}: fixed residues not modeled in structure: {missing}")

    fixed_idx = sorted(num_to_idx[rn] for rn in fixed_residue_numbers)

    # Hard gate: the frozen positions must really be the catalytic triad.
    for rn, expected in TRIAD_EXPECTED.items():
        if rn in num_to_idx:
            found = wildtype_seq[num_to_idx[rn]]
            if found != expected:
                raise ValueError(
                    f"{pdb_id}: residue {rn} is {found}, expected {expected}. "
                    "This is not the IsPETase catalytic triad numbering."
                )

    fixed_set = set(fixed_idx)
    mutable_idx = [i for i in range(len(wildtype_seq)) if i not in fixed_set]

    return Target(
        pdb_id=pdb_id,
        chain=chain,
        wildtype_seq=wildtype_seq,
        residue_numbers=residue_numbers,
        fixed_residue_numbers=tuple(fixed_residue_numbers),
        fixed_idx=fixed_idx,
        mutable_idx=mutable_idx,
    )


def wildtype_vector(target: Target) -> np.ndarray:
    """The optimizer vector that decodes back to the unmutated sequence."""
    return np.array(
        [AA_TO_IDX[target.wildtype_seq[i]] for i in target.mutable_idx], dtype=int
    )


def decode(target: Target, x: np.ndarray) -> str:
    """Integer vector -> full-length sequence, with fixed positions untouched."""
    seq = list(target.wildtype_seq)
    for idx, aa_i in zip(target.mutable_idx, x):
        seq[idx] = AA_ALPHABET[int(aa_i)]
    return "".join(seq)


def decode_many(target: Target, X: np.ndarray) -> list[str]:
    return [decode(target, x) for x in np.atleast_2d(X)]


def hamming(target: Target, seq: str) -> int:
    return sum(a != b for a, b in zip(seq, target.wildtype_seq))


def mutations(target: Target, seq: str) -> list[tuple[int, str, str]]:
    """List of (pdb_residue_number, wildtype_aa, mutant_aa)."""
    return [
        (target.resnum(i), target.wildtype_seq[i], seq[i])
        for i in range(len(seq))
        if seq[i] != target.wildtype_seq[i]
    ]


def format_mutations(target: Target, seq: str) -> str:
    """Compact literature-style notation, e.g. 'S121E N233K'."""
    return " ".join(f"{wt}{rn}{mt}" for rn, wt, mt in mutations(target, seq))

In [ ]:
%%writefile polyphage/scoring.py
"""ProteinMPNN scoring: foldability as negative log-likelihood of a sequence
given a fixed backbone.

Two implementations:

* ``SubprocessScorer`` -- shells out to ``protein_mpnn_run.py --score_only 1``,
  once per sequence. Slow (reloads weights every call) but it is ProteinMPNN's
  own code path, so it is the reference the fast one is checked against.
* ``BatchScorer`` -- loads the model once, featurizes the backbone once, and
  scores a whole population in one forward pass.

Never trust ``BatchScorer`` until ``check_equivalence`` passes.
"""

from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import numpy as np

# ProteinMPNN's own alphabet, including the 'X' slot. Order matters: it is the
# index space the model's log-probs are in.
MPNN_ALPHABET = "ACDEFGHIKLMNPQRSTVWYX"


class SubprocessScorer:
    """Reference scorer. Correct by construction, far too slow to optimize with."""

    def __init__(
        self,
        mpnn_dir: str | Path,
        parsed_jsonl: str | Path,
        fixed_jsonl: str | Path,
        out_dir: str | Path = "outputs/score_subprocess",
    ):
        self.mpnn_dir = Path(mpnn_dir)
        self.parsed_jsonl = Path(parsed_jsonl).resolve()
        self.fixed_jsonl = Path(fixed_jsonl).resolve()
        self.out_dir = Path(out_dir).resolve()
        self.out_dir.mkdir(parents=True, exist_ok=True)

    def score(self, seq: str, seed: int | None = None) -> float:
        """Score one sequence.

        ``seed`` is passed through to protein_mpnn_run.py, which calls
        torch.manual_seed with it. Without a seed the script picks a random one,
        so the decoding order -- and therefore the score -- differs per call.
        """
        fasta = self.out_dir / "candidate.fasta"
        fasta.write_text(f">candidate\n{seq}\n")

        cmd = [
            sys.executable, "protein_mpnn_run.py",
            "--jsonl_path", str(self.parsed_jsonl),
            "--fixed_positions_jsonl", str(self.fixed_jsonl),
            "--out_folder", str(self.out_dir),
            "--score_only", "1",
            "--save_score", "1",
            "--path_to_fasta", str(fasta),
        ]
        if seed is not None:
            cmd += ["--seed", str(seed)]

        result = subprocess.run(
            cmd, cwd=self.mpnn_dir, capture_output=True, text=True
        )
        if result.returncode != 0:
            raise RuntimeError(
                "protein_mpnn_run.py failed\n"
                f"stdout:\n{result.stdout[-2000:]}\n\nstderr:\n{result.stderr[-2000:]}"
            )

        npz_files = sorted((self.out_dir / "score_only").glob("*.npz"))
        if not npz_files:
            raise RuntimeError(f"no .npz written under {self.out_dir / 'score_only'}")
        newest = max(npz_files, key=lambda p: p.stat().st_mtime)
        return float(np.load(newest)["score"].mean())


class BatchScorer:
    """Loads ProteinMPNN once and scores a population per forward pass.

    The backbone features are computed a single time in ``__init__`` and reused,
    which is the whole point: for a fixed-backbone design problem they never
    change, only the sequence does.
    """

    def __init__(
        self,
        mpnn_dir: str | Path,
        pdb_path: str | Path,
        chain: str = "A",
        weights: str = "v_48_020.pt",
        device: str | None = None,
        seed: int = 0,
        n_decoding_orders: int = 8,
    ):
        import torch

        self.mpnn_dir = Path(mpnn_dir)
        if str(self.mpnn_dir) not in sys.path:
            sys.path.insert(0, str(self.mpnn_dir))

        from protein_mpnn_utils import (
            ProteinMPNN, _scores, parse_PDB, tied_featurize,
        )

        self.torch = torch
        self._scores = _scores
        self.device = torch.device(
            device if device is not None
            else ("cuda" if torch.cuda.is_available() else "cpu")
        )
        self.seed = seed
        self.n_decoding_orders = n_decoding_orders

        ckpt_path = self.mpnn_dir / "vanilla_model_weights" / weights
        if not ckpt_path.exists():
            raise FileNotFoundError(f"weights not found: {ckpt_path}")
        ckpt = torch.load(ckpt_path, map_location=self.device, weights_only=False)

        self.model = ProteinMPNN(
            ca_only=False,
            num_letters=21,
            node_features=128,
            edge_features=128,
            hidden_dim=128,
            num_encoder_layers=3,
            num_decoder_layers=3,
            augment_eps=0.0,
            k_neighbors=ckpt["num_edges"],
        )
        self.model.load_state_dict(ckpt["model_state_dict"])
        self.model.to(self.device)
        self.model.eval()

        # Featurize the backbone once.
        pdb_dict_list = parse_PDB(str(pdb_path), ca_only=False)
        chain_id_dict = {pdb_dict_list[0]["name"]: ([chain], [])}
        feat = tied_featurize(
            pdb_dict_list, self.device, chain_id_dict,
            None, None, None, None, None, ca_only=False,
        )
        # tied_featurize returns a long positional tuple; these indices follow
        # protein_mpnn_run.py. If ProteinMPNN is upgraded, re-check them -- the
        # equivalence test below is what catches a silent reshuffle.
        self._X = feat[0]
        self._mask = feat[2]
        self._chain_M = feat[4]
        self._chain_encoding_all = feat[5]
        self._chain_M_pos = feat[10]
        self._residue_idx = feat[12]
        self.seq_len = int(self._X.shape[1])

    def _encode(self, seqs: list[str]) -> "object":
        torch = self.torch
        idx = {aa: i for i, aa in enumerate(MPNN_ALPHABET)}
        rows = []
        for seq in seqs:
            if len(seq) != self.seq_len:
                raise ValueError(
                    f"sequence length {len(seq)} != backbone length {self.seq_len}"
                )
            rows.append([idx.get(aa, idx["X"]) for aa in seq])
        return torch.tensor(rows, dtype=torch.long, device=self.device)

    def score(
        self,
        seqs: list[str] | str,
        n_orders: int | None = None,
        seed: int | None = None,
    ) -> np.ndarray:
        """Mean per-residue negative log-likelihood. Lower = model likes it more.

        ProteinMPNN scores a sequence under a *random* autoregressive decoding
        order, so a single draw is a noisy estimate (spread of ~0.02 nats on
        IsPETase). Two consequences:

        * The score is averaged over ``n_orders`` decoding orders, drawn from a
          fixed seed sequence. That makes it deterministic across calls (so the
          optimizer compares candidates on equal terms) and cuts the noise by
          ~sqrt(n_orders). Without this, NSGA-II optimizes the dice: the
          per-draw noise is the same size as the score change from one or two
          mutations.
        * ``n_orders=1`` with an explicit ``seed`` reproduces one specific
          decoding order, which is what the equivalence check needs.
        """
        torch = self.torch
        if isinstance(seqs, str):
            seqs = [seqs]
        if not seqs:
            return np.zeros(0)

        n_orders = self.n_decoding_orders if n_orders is None else n_orders
        base_seed = self.seed if seed is None else seed

        n = len(seqs)
        S = self._encode(seqs)
        tile = lambda t: t.expand(n, *t.shape[1:]).contiguous()  # noqa: E731
        X = tile(self._X)
        mask = tile(self._mask)
        chain_M = tile(self._chain_M)
        chain_M_pos = tile(self._chain_M_pos)
        chain_encoding_all = tile(self._chain_encoding_all)
        residue_idx = tile(self._residue_idx)
        mask_for_loss = mask * chain_M * chain_M_pos

        total = np.zeros(n)
        with torch.no_grad():
            for k in range(n_orders):
                # Seed the same generator protein_mpnn_run.py uses (the global
                # one, on this device), so a matching seed gives a matching
                # decoding order.
                torch.manual_seed(base_seed + k)
                randn = torch.randn(chain_M.shape, device=self.device)
                log_probs = self.model(
                    X, S, mask, chain_M * chain_M_pos,
                    residue_idx, chain_encoding_all, randn,
                )
                total += self._scores(S, log_probs, mask_for_loss).cpu().numpy()

        return (total / n_orders).astype(float)


def check_equivalence(
    batch: BatchScorer,
    reference: SubprocessScorer,
    seq: str,
    seed: int = 37,
    tol: float = 1e-4,
    n_probe: int = 6,
) -> dict:
    """Assert the fast scorer agrees with ProteinMPNN's own script.

    Run this on the wildtype before any optimization. A real mismatch means
    ``BatchScorer`` is scoring something other than what it claims, which no
    amount of downstream tuning will fix.

    The comparison forces both scorers onto the *same* decoding order by seeding
    them identically, so the scores should agree to float tolerance. Comparing
    unseeded calls is meaningless: each draws its own random decoding order, and
    the resulting spread (~0.02 nats here) swamps any tolerance tight enough to
    catch a genuine bug.

    Caveat: the same-seed trick relies on both processes drawing ``randn`` from
    the same RNG stream. That holds on GPU, where the draw is the first use of
    the CUDA generator after ``manual_seed``. On CPU it can diverge, because
    building the model consumes CPU randomness first -- hence the statistical
    fallback below.
    """
    fast = float(batch.score([seq], n_orders=1, seed=seed)[0])
    slow = reference.score(seq, seed=seed)
    delta = abs(fast - slow)
    print(f"same decoding order (seed={seed}):")
    print(f"  batch={fast:.6f}  subprocess={slow:.6f}  |delta|={delta:.6f}")

    if delta <= tol:
        print("  PASS -- featurization matches ProteinMPNN exactly.")
        return {"mode": "exact", "delta": delta, "passed": True}

    # Not an exact match. Decide whether it is a feature bug or an RNG-stream
    # difference, by checking whether the reference value is an ordinary member
    # of the batch scorer's own across-order distribution.
    probe = np.array(
        [float(batch.score([seq], n_orders=1, seed=s)[0]) for s in range(n_probe)]
    )
    lo, hi, sd = probe.min(), probe.max(), probe.std()
    print(f"\n  batch across {n_probe} decoding orders:")
    print(f"    mean={probe.mean():.6f}  sd={sd:.6f}  range=[{lo:.6f}, {hi:.6f}]")

    within = lo - 2 * sd <= slow <= hi + 2 * sd
    if within:
        print(
            "\n  LIKELY OK -- the subprocess value sits inside the batch scorer's\n"
            "  own across-order spread, so the backbone features agree and only the\n"
            "  decoding order differs (different RNG stream). Averaging over\n"
            f"  n_decoding_orders={batch.n_decoding_orders} is what makes the\n"
            "  objective stable; proceed, but note this in the report."
        )
        return {
            "mode": "statistical", "delta": delta, "passed": True,
            "subprocess": slow, "batch_mean": float(probe.mean()), "batch_sd": float(sd),
        }

    raise AssertionError(
        f"subprocess score {slow:.6f} lies outside the batch scorer's across-order "
        f"range [{lo:.6f}, {hi:.6f}] (sd={sd:.6f}). This is a real mismatch, not "
        "decoding-order noise -- re-check the tied_featurize indices in "
        "BatchScorer.__init__ against this clone's protein_mpnn_utils.py."
    )

In [ ]:
%%writefile polyphage/optimize.py
"""NSGA-II over PETase sequences.

Design notes worth defending in the report:

* Amino-acid identity is **categorical**, not ordinal. pymoo's default SBX
  crossover and polynomial mutation assume a continuous, ordered variable --
  under them "index 5 (F) is between 4 (E) and 6 (G)", which is biochemically
  meaningless and biases the search toward whatever happens to sit mid-alphabet.
  This module uses uniform crossover and random-reset mutation instead.
* Mutation can also *revert* a position to wildtype. Without that, mutation
  count only ever ratchets upward and the low-mutation end of the Pareto front
  collapses.
* A hard constraint caps Hamming distance from wildtype, so candidates stay in
  the experimentally testable regime (real variants: 5-10 mutations) no matter
  how long the run goes.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.core.mutation import Mutation
from pymoo.core.problem import Problem
from pymoo.core.sampling import Sampling
from pymoo.operators.crossover.ux import UniformCrossover
from pymoo.optimize import minimize

from .encoding import decode, hamming, wildtype_vector

N_AA = 20


class WildtypeBiasedSampling(Sampling):
    """Seed the population at wildtype, then perturb a handful of positions."""

    def __init__(self, target, min_mut: int = 1, max_mut: int = 8):
        super().__init__()
        self.target = target
        self.min_mut = min_mut
        self.max_mut = max_mut

    def _do(self, problem, n_samples, **kwargs):
        wt = wildtype_vector(self.target)
        X = np.tile(wt, (n_samples, 1))
        rng = np.random.default_rng(kwargs.get("seed"))
        for row in X:
            k = rng.integers(self.min_mut, self.max_mut + 1)
            where = rng.choice(len(row), size=k, replace=False)
            row[where] = rng.integers(0, N_AA, size=k)
        return X


class RandomResetMutation(Mutation):
    """Per-gene categorical mutation.

    With probability ``prob`` a gene is touched. A touched gene reverts to
    wildtype with probability ``revert_prob``, otherwise it is resampled
    uniformly over the 20 amino acids.
    """

    def __init__(self, target, prob: float = 0.01, revert_prob: float = 0.5):
        super().__init__()
        self.target = target
        self.gene_prob = prob
        self.revert_prob = revert_prob

    def _do(self, problem, X, **kwargs):
        X = X.astype(int).copy()
        wt = wildtype_vector(self.target)
        rng = np.random.default_rng(kwargs.get("seed"))

        touched = rng.random(X.shape) < self.gene_prob
        revert = touched & (rng.random(X.shape) < self.revert_prob)
        resample = touched & ~revert

        X[revert] = np.broadcast_to(wt, X.shape)[revert]
        X[resample] = rng.integers(0, N_AA, size=int(resample.sum()))
        return X


@dataclass
class ObjectiveSpec:
    """One scalar objective. ``fn`` takes a list of sequences, returns an array.

    All objectives are *minimized*; negate inside ``fn`` for things you want to
    maximize (as novelty does).
    """

    name: str
    fn: callable


class PETaseProblem(Problem):
    def __init__(self, target, objectives: list[ObjectiveSpec], max_mutations: int = 15):
        self.target = target
        self.objectives = objectives
        self.max_mutations = max_mutations
        super().__init__(
            n_var=target.n_mutable,
            n_obj=len(objectives),
            n_ieq_constr=1,
            xl=0,
            xu=N_AA - 1,
            vtype=int,
        )

    def _evaluate(self, X, out, *args, **kwargs):
        seqs = [decode(self.target, x) for x in np.atleast_2d(X)]
        out["F"] = np.column_stack([spec.fn(seqs) for spec in self.objectives])
        dists = np.array([hamming(self.target, s) for s in seqs], dtype=float)
        # g <= 0 is feasible
        out["G"] = (dists - self.max_mutations).reshape(-1, 1)


def foldability_objective(scorer) -> ObjectiveSpec:
    """ProteinMPNN negative log-likelihood. Lower is better, so minimize directly."""
    return ObjectiveSpec("mpnn_nll", lambda seqs: np.asarray(scorer.score(seqs), dtype=float))


def novelty_objective(target) -> ObjectiveSpec:
    """Mutation count. Maximized, hence negated.

    This is a weak objective -- a proxy for "explores further from wildtype",
    not a property anyone wants in an enzyme. It exists to give the front a
    second axis until a real one (pLDDT, ddG, docking) replaces it.
    """
    return ObjectiveSpec(
        "neg_n_mutations",
        lambda seqs: -np.array([hamming(target, s) for s in seqs], dtype=float),
    )


def run(
    target,
    objectives: list[ObjectiveSpec],
    pop_size: int = 8,
    n_gen: int = 3,
    max_mutations: int = 15,
    mutation_prob: float | None = None,
    seed: int = 1,
    verbose: bool = True,
):
    """Run NSGA-II. Returns the pymoo result object.

    ``mutation_prob`` defaults to roughly two touched genes per individual per
    generation, which keeps the search local on a ~265-residue protein.
    """
    if mutation_prob is None:
        mutation_prob = 2.0 / target.n_mutable

    problem = PETaseProblem(target, objectives, max_mutations=max_mutations)
    algorithm = NSGA2(
        pop_size=pop_size,
        sampling=WildtypeBiasedSampling(target, max_mut=min(8, max_mutations)),
        crossover=UniformCrossover(prob=0.9),
        mutation=RandomResetMutation(target, prob=mutation_prob),
        eliminate_duplicates=True,
    )
    return minimize(
        problem, algorithm, ("n_gen", n_gen), seed=seed, verbose=verbose,
        save_history=True,
    )

In [ ]:
%%writefile polyphage/validate.py
"""Validation and reporting: does the search find what the literature already knows?

Hotspot overlap is the project's main internal validity check. It is not proof
the designs work -- it is evidence the search is finding biologically real
signal rather than noise, which is exactly what a purely computational project
can honestly claim.
"""

from __future__ import annotations

import json
import time
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np

from .encoding import format_mutations, hamming, mutations

# Lu et al. 2022, Nature -- FAST-PETase, 5 mutations vs wildtype IsPETase.
FAST_PETASE = {121: "E", 186: "H", 224: "Q", 233: "K", 280: "A"}

# Cui et al. 2021 -- DuraPETase, 10 mutations.
DURA_PETASE = {
    117: "F", 119: "Y", 140: "D", 159: "H", 165: "A",
    168: "R", 180: "I", 188: "Q", 214: "H", 280: "A",
}

KNOWN_HOTSPOTS = set(FAST_PETASE) | set(DURA_PETASE)


@dataclass
class VariantReport:
    sequence: str
    objectives: dict[str, float]
    n_mutations: int
    mutation_str: str
    hotspot_positions: list[int] = field(default_factory=list)
    exact_matches: list[str] = field(default_factory=list)


def report_front(target, res, objective_names: list[str]) -> list[VariantReport]:
    """Turn a pymoo result into readable, checkable variant records."""
    from .encoding import decode

    X = np.atleast_2d(res.X)
    F = np.atleast_2d(res.F)
    reports = []

    for x, f in zip(X, F):
        seq = decode(target, x)
        muts = mutations(target, seq)
        positions = {rn for rn, _, _ in muts}

        exact = []
        for rn, _, mt in muts:
            for name, table in (("FAST", FAST_PETASE), ("Dura", DURA_PETASE)):
                if table.get(rn) == mt:
                    exact.append(f"{name}:{rn}{mt}")

        reports.append(
            VariantReport(
                sequence=seq,
                objectives=dict(zip(objective_names, (float(v) for v in f))),
                n_mutations=hamming(target, seq),
                mutation_str=format_mutations(target, seq),
                hotspot_positions=sorted(positions & KNOWN_HOTSPOTS),
                exact_matches=sorted(set(exact)),
            )
        )

    return sorted(reports, key=lambda r: tuple(r.objectives.values()))


def print_front(reports: list[VariantReport], max_mutations_shown: int = 20) -> None:
    print(f"{len(reports)} Pareto-optimal variants\n")
    for i, r in enumerate(reports, 1):
        objs = "  ".join(f"{k}={v:.4f}" for k, v in r.objectives.items())
        print(f"[{i}] {objs}  n_mut={r.n_mutations}")
        shown = r.mutation_str.split()
        if len(shown) > max_mutations_shown:
            print(f"    {' '.join(shown[:max_mutations_shown])} ... (+{len(shown) - max_mutations_shown} more)")
        else:
            print(f"    {r.mutation_str or '(wildtype)'}")
        if r.hotspot_positions:
            print(f"    known hotspot positions hit: {r.hotspot_positions}")
        if r.exact_matches:
            print(f"    EXACT literature mutations: {r.exact_matches}")
        print()


def summarize(reports: list[VariantReport]) -> dict:
    """One-line-per-run metrics. This is what goes in the run log."""
    n_muts = [r.n_mutations for r in reports]
    hit_positions = sorted({p for r in reports for p in r.hotspot_positions})
    exact = sorted({m for r in reports for m in r.exact_matches})
    return {
        "n_variants": len(reports),
        "n_mutations_min": min(n_muts) if n_muts else 0,
        "n_mutations_max": max(n_muts) if n_muts else 0,
        "hotspot_positions_hit": hit_positions,
        "n_hotspot_positions_hit": len(hit_positions),
        "exact_literature_mutations": exact,
    }


def log_run(
    config: dict,
    reports: list[VariantReport],
    wall_seconds: float,
    path: str | Path = "runs.jsonl",
) -> dict:
    """Append a run record. The progression across runs is the report narrative;
    it cannot be reconstructed later if runs are not logged as they happen."""
    record = {
        "timestamp": time.strftime("%Y-%m-%dT%H:%M:%S"),
        "wall_seconds": round(wall_seconds, 1),
        "config": config,
        "results": summarize(reports),
        "variants": [
            {"mutations": r.mutation_str, "objectives": r.objectives, "n_mut": r.n_mutations}
            for r in reports
        ],
    }
    with open(path, "a") as f:
        f.write(json.dumps(record) + "\n")
    return record


def plot_front(reports: list[VariantReport], wildtype_objectives: dict | None = None):
    """Scatter the first two objectives.

    A front that is a single point or a straight line means the objectives are
    not in genuine conflict and one of them is doing no work.
    """
    import matplotlib.pyplot as plt

    names = list(reports[0].objectives)
    if len(names) < 2:
        raise ValueError("need at least 2 objectives to plot a front")

    xs = [r.objectives[names[0]] for r in reports]
    ys = [r.objectives[names[1]] for r in reports]

    fig, ax = plt.subplots(figsize=(6, 4.5))
    ax.scatter(xs, ys, s=60, label="Pareto front", zorder=3)
    for r, x, y in zip(reports, xs, ys):
        ax.annotate(f"{r.n_mutations}", (x, y), textcoords="offset points",
                    xytext=(6, 4), fontsize=8)

    if wildtype_objectives:
        ax.scatter([wildtype_objectives[names[0]]], [wildtype_objectives[names[1]]],
                   marker="*", s=220, color="crimson", label="wildtype", zorder=4)

    ax.set_xlabel(names[0])
    ax.set_ylabel(names[1])
    ax.set_title("Polyphage Pareto front (labels = mutation count)")
    ax.legend()
    fig.tight_layout()
    return fig, ax

In [ ]:
%%writefile polyphage/control.py
"""Randomization control: is the hotspot overlap better than chance?

Run 2 recovered T140D, an exact DuraPETase mutation. That is only evidence if
random mutations of the same size do not recover it nearly as often. With 15
known hotspot positions among ~262 mutable ones, a 9-mutation variant lands on
*some* hotspot position maybe 40% of the time by luck, so the position-level
overlap is weak evidence on its own. Matching the exact amino acid is much
rarer. This module measures both against an explicit null instead of guessing.

The null used here is uniform random mutation: same number of mutations, random
positions, random replacement amino acids. That is the honest baseline for the
claim "the search found these, not chance". A stricter null -- random variants
resampled from ProteinMPNN's own high-likelihood region -- would test the
stronger claim "the *optimizer* found these, not just the model's prior", and
is worth adding once this one is reported.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np

from .encoding import AA_ALPHABET
from .validate import DURA_PETASE, FAST_PETASE, KNOWN_HOTSPOTS, VariantReport


@dataclass
class NullResult:
    n_mutations: int
    n_samples: int
    observed_positions: int
    observed_exact: int
    null_positions_mean: float
    null_exact_mean: float
    p_positions: float
    p_exact: float

    def __str__(self) -> str:
        return (
            f"{self.n_mutations} mutations, {self.n_samples} random draws\n"
            f"  hotspot positions hit : observed {self.observed_positions}, "
            f"chance {self.null_positions_mean:.2f}  ->  p = {self.p_positions:.4f}\n"
            f"  exact literature hits : observed {self.observed_exact}, "
            f"chance {self.null_exact_mean:.3f}  ->  p = {self.p_exact:.4f}"
        )


def _exact_tables() -> dict[int, str]:
    """Position -> amino acid, for every experimentally confirmed mutation."""
    combined = dict(DURA_PETASE)
    combined.update(FAST_PETASE)
    return combined


def sample_random_variants(
    target, n_mutations: int, n_samples: int, rng: np.random.Generator
) -> tuple[np.ndarray, np.ndarray]:
    """Draw random variants; return (hotspot position hits, exact matches) counts.

    Mutations are drawn without replacement over mutable positions, and the
    replacement amino acid is never the wildtype one -- otherwise a "mutation"
    could silently be a no-op and the null would be too easy to beat.
    """
    exact = _exact_tables()
    mutable_resnums = np.array([target.resnum(i) for i in target.mutable_idx])
    wt_aa = np.array([target.wildtype_seq[i] for i in target.mutable_idx])

    hotspot_mask = np.array([rn in KNOWN_HOTSPOTS for rn in mutable_resnums])

    pos_hits = np.zeros(n_samples, dtype=int)
    exact_hits = np.zeros(n_samples, dtype=int)

    n_mutable = len(mutable_resnums)
    for s in range(n_samples):
        picks = rng.choice(n_mutable, size=n_mutations, replace=False)
        pos_hits[s] = int(hotspot_mask[picks].sum())

        n_exact = 0
        for p in picks:
            choices = [a for a in AA_ALPHABET if a != wt_aa[p]]
            new_aa = choices[rng.integers(len(choices))]
            if exact.get(int(mutable_resnums[p])) == new_aa:
                n_exact += 1
        exact_hits[s] = n_exact

    return pos_hits, exact_hits


def compare_to_null(
    target,
    report: VariantReport,
    n_samples: int = 5000,
    seed: int = 0,
) -> NullResult:
    """Empirical p-values for one variant's hotspot overlap."""
    rng = np.random.default_rng(seed)
    k = report.n_mutations
    pos_hits, exact_hits = sample_random_variants(target, k, n_samples, rng)

    obs_pos = len(report.hotspot_positions)
    obs_exact = len(report.exact_matches)

    # One-sided: how often does chance do at least as well as observed?
    # +1 in numerator and denominator keeps p away from exactly 0, which an
    # empirical estimate can never justify.
    p_pos = (np.sum(pos_hits >= obs_pos) + 1) / (n_samples + 1)
    p_exact = (np.sum(exact_hits >= obs_exact) + 1) / (n_samples + 1)

    return NullResult(
        n_mutations=k,
        n_samples=n_samples,
        observed_positions=obs_pos,
        observed_exact=obs_exact,
        null_positions_mean=float(pos_hits.mean()),
        null_exact_mean=float(exact_hits.mean()),
        p_positions=float(p_pos),
        p_exact=float(p_exact),
    )


def run_control(
    target, reports: list[VariantReport], n_samples: int = 5000, seed: int = 0
) -> list[NullResult]:
    """Control every variant on the front, and print a verdict."""
    results = [compare_to_null(target, r, n_samples, seed) for r in reports]

    print(f"Randomization control ({n_samples} random variants per mutation count)\n")
    for r, res in zip(reports, results):
        print(f"[{r.mutation_str}]")
        print(res)
        print()

    best_exact = min((r.p_exact for r in results), default=1.0)
    any_exact = any(r.observed_exact > 0 for r in results)

    if any_exact and best_exact < 0.05:
        print(
            f"VERDICT: exact literature matches are unlikely by chance "
            f"(best p = {best_exact:.4f}). This is a defensible result."
        )
    elif any_exact:
        print(
            f"VERDICT: exact matches found, but chance alone reproduces them at "
            f"p = {best_exact:.4f}. Report the match, but do not claim the method "
            "beat chance on this evidence -- run more generations or more seeds."
        )
    else:
        print(
            "VERDICT: no exact literature matches on this front. Position-level "
            "overlap alone is weak evidence; do not lead the report with it."
        )

    return results


def repeat_seeds_summary(per_seed_reports: dict[int, list[VariantReport]]) -> dict:
    """Aggregate hotspot recovery across several optimizer seeds.

    A single run recovering T140D could be luck in the search itself, separate
    from luck in the null. Running the same config under several seeds and
    reporting how many recover it is the cheapest way to show the result is a
    property of the method and not of seed 1.
    """
    found = {}
    for seed, reports in per_seed_reports.items():
        matches = {m for r in reports for m in r.exact_matches}
        found[seed] = sorted(matches)

    all_matches = sorted({m for ms in found.values() for m in ms})
    recovery = {
        m: sum(1 for ms in found.values() if m in ms) / len(found)
        for m in all_matches
    }
    return {"per_seed": found, "recovery_rate": recovery, "n_seeds": len(found)}

In [ ]:
%%writefile polyphage/stability.py
"""Thermostability objective via ThermoMPNN.

ProteinMPNN's score answers "does this sequence fit this backbone". It says
nothing about heat. ThermoMPNN (Kuhlman Lab, MIT licence) answers the stability
question directly: it reuses ProteinMPNN's residue embeddings and adds a head
trained on the Megascale dataset to predict ddG, the change in folding free
energy on mutation. **Negative ddG = stabilising.**

The efficient trick: ThermoMPNN's ``custom_inference.py`` runs *site-saturation
mutagenesis* -- every position against every amino acid in one pass. Run it once
on the wildtype backbone, cache the resulting table, and scoring a variant
inside the optimiser becomes a handful of dictionary lookups. No model calls in
the loop at all.

LIMITATION, to state plainly in the report: ThermoMPNN is trained on *single*
point mutants. Summing per-mutation ddG over a 9-mutation variant assumes the
mutations do not interact. They do -- that interaction is called epistasis, and
additive models generally overestimate how stabilising a mutation stack is. This
is a standard, published screening heuristic, not a ground truth. Its job is to
rank candidates cheaply; the top few should later be checked with a method that
models the whole variant at once (ThermoMPNN-D handles pairs; MD or FoldX on the
folded structure handles the rest).
"""

from __future__ import annotations

import subprocess
import sys
from pathlib import Path

import numpy as np

from .encoding import AA_ALPHABET, mutations


class StabilityOracle:
    """Cached site-saturation ddG table.

    Builds once from ThermoMPNN, then answers any variant by lookup.
    """

    def __init__(
        self,
        thermompnn_dir: str | Path,
        pdb_path: str | Path,
        target,
        chain: str = "A",
        cache_csv: str | Path = "outputs/ssm_ddg.csv",
        python: str | None = None,
    ):
        self.dir = Path(thermompnn_dir)
        self.pdb_path = Path(pdb_path).resolve()
        self.target = target
        self.chain = chain
        self.cache_csv = Path(cache_csv).resolve()
        self.python = python or sys.executable
        self.table: dict[tuple[int, str], float] = {}

    def build(self, force: bool = False, extra_args: list[str] | None = None) -> None:
        """Run ThermoMPNN's site-saturation scan once and parse it.

        ``custom_inference.py``'s exact flag names are not documented in the
        README, so run ``!python custom_inference.py -h`` in the notebook first
        and pass anything extra through ``extra_args`` if the defaults below do
        not match this checkout.
        """
        if not self.cache_csv.exists() or force:
            self.cache_csv.parent.mkdir(parents=True, exist_ok=True)
            cmd = [
                self.python, "custom_inference.py",
                "--pdb", str(self.pdb_path),
                "--chain", self.chain,
                "--out", str(self.cache_csv),
            ]
            if extra_args:
                cmd += extra_args
            result = subprocess.run(
                cmd, cwd=self.dir, capture_output=True, text=True
            )
            if result.returncode != 0:
                raise RuntimeError(
                    "ThermoMPNN custom_inference.py failed. Run "
                    "`!python custom_inference.py -h` in the ThermoMPNN directory "
                    "and adjust the flags in StabilityOracle.build.\n"
                    f"stdout:\n{result.stdout[-2000:]}\n\nstderr:\n{result.stderr[-2000:]}"
                )

        self.table = self._parse(self.cache_csv)
        self._validate_coverage()

    def _parse(self, csv_path: Path) -> dict[tuple[int, str], float]:
        """Parse the SSM output into {(residue_number, mutant_aa): ddG}.

        Column names vary between ThermoMPNN versions, so they are matched by
        likely aliases rather than assumed.
        """
        import pandas as pd

        df = pd.read_csv(csv_path)
        cols = {c.lower().strip(): c for c in df.columns}

        def pick(*aliases, required=True):
            for a in aliases:
                if a in cols:
                    return cols[a]
            if required:
                raise ValueError(
                    f"could not find a column among {aliases} in {csv_path}. "
                    f"Columns present: {list(df.columns)}. Adjust "
                    "StabilityOracle._parse."
                )
            return None

        pos_col = pick("position", "pos", "resnum", "residue", "pdb_position")
        mut_col = pick("mutant", "mut", "mutation", "mut_aa", "mutant_aa")
        ddg_col = pick("ddg", "ddg_pred", "prediction", "predicted_ddg", "ddG")

        table: dict[tuple[int, str], float] = {}
        for _, row in df.iterrows():
            mut = str(row[mut_col]).strip()
            # A "mutation" column may be either a bare amino acid ("D") or full
            # notation ("T140D"); take the final character either way.
            aa = mut[-1].upper()
            if aa not in AA_ALPHABET:
                continue
            try:
                resnum = int(row[pos_col])
            except (TypeError, ValueError):
                continue
            table[(resnum, aa)] = float(row[ddg_col])
        return table

    def _validate_coverage(self) -> None:
        """Fail loudly if the table does not cover the design positions."""
        if not self.table:
            raise ValueError(f"no usable rows parsed from {self.cache_csv}")

        wanted = {self.target.resnum(i) for i in self.target.mutable_idx}
        covered = {rn for rn, _ in self.table}
        missing = wanted - covered
        frac = 1 - len(missing) / len(wanted)
        print(
            f"ddG table: {len(self.table)} entries, "
            f"{frac:.1%} of mutable positions covered"
        )
        if frac < 0.9:
            raise ValueError(
                f"ddG table covers only {frac:.1%} of mutable positions "
                f"({len(missing)} missing). Residue numbering probably differs "
                "between ThermoMPNN's output and the design target -- check "
                f"a few missing numbers: {sorted(missing)[:10]}"
            )

    def ddg(self, seq: str) -> float:
        """Additive ddG for a variant. Negative = predicted more stable."""
        total = 0.0
        for resnum, _wt, mut in mutations(self.target, seq):
            value = self.table.get((resnum, mut))
            if value is None:
                # Unknown mutation: treat as neutral rather than silently
                # rewarding it. Coverage is validated at build time, so this
                # should be rare.
                continue
            total += value
        return total

    def ddg_many(self, seqs: list[str]) -> np.ndarray:
        return np.array([self.ddg(s) for s in seqs], dtype=float)

    def best_stabilising(self, n: int = 20) -> list[tuple[int, str, float]]:
        """The n most stabilising single mutations ThermoMPNN predicts.

        Useful on its own: compare this list against the FAST-PETase and
        DuraPETase mutation sets before running any search. If ThermoMPNN
        already ranks the known mutations highly, the optimiser is starting from
        a well-informed prior and that is worth reporting.
        """
        items = sorted(self.table.items(), key=lambda kv: kv[1])[:n]
        return [(rn, aa, v) for (rn, aa), v in items]


def stability_objective(oracle: StabilityOracle):
    """ObjectiveSpec for ddG. Minimised, because negative ddG is stabilising."""
    from .optimize import ObjectiveSpec

    return ObjectiveSpec("ddg", lambda seqs: oracle.ddg_many(seqs))

## Cell 3 — Fetch and parse the structure

In [ ]:
import urllib.request, shutil

# 6EQE = wild-type IsPETase (0.92 A).  7SH6 = FAST-PETase (reference only).
for pdb in ["6EQE", "7SH6"]:
    urllib.request.urlretrieve(f"https://files.rcsb.org/download/{pdb}.pdb",
                               f"inputs/pdbs/{pdb}.pdb")
    print("downloaded", pdb)

shutil.copy("inputs/pdbs/6EQE.pdb", "inputs/design_target/6EQE.pdb")

In [ ]:
!python helper_scripts/parse_multiple_chains.py \
  --input_path=inputs/design_target/ \
  --output_path=outputs/parsed_6EQE.jsonl

!python helper_scripts/make_fixed_positions_dict.py \
  --input_path=outputs/parsed_6EQE.jsonl \
  --output_path=outputs/fixed_6EQE.jsonl \
  --chain_list "A" \
  --position_list "160 206 237"

## Cell 4 — Load the target (GATE)

`load_target` raises if the triad is not SER/ASP/HIS at 160/206/237, or if
Biopython and ProteinMPNN disagree about the sequence residue by residue.
If it raises, **stop** -- every residue number downstream would be wrong.

In [ ]:
from polyphage import load_target

target = load_target(
    pdb_path="inputs/design_target/6EQE.pdb",
    parsed_jsonl="outputs/parsed_6EQE.jsonl",
    chain="A",
)
print("length:", len(target.wildtype_seq))
print("modeled residues:", target.residue_numbers[0], "-", target.residue_numbers[-1])
print("mutable positions:", target.n_mutable)
print("frozen (triad):", [(rn, target.wildtype_seq[i])
                          for rn, i in zip(target.fixed_residue_numbers, target.fixed_idx)])

## Cell 5 — Scorers

`BatchScorer` loads the model and featurizes the backbone once, then scores a
whole population per forward pass. `SubprocessScorer` is ProteinMPNN's own
`--score_only` path -- slow, but it is the reference the fast one is checked against.

In [ ]:
from polyphage import BatchScorer, SubprocessScorer, check_equivalence

batch = BatchScorer(mpnn_dir=".", pdb_path="inputs/design_target/6EQE.pdb", chain="A")
ref   = SubprocessScorer(mpnn_dir=".",
                         parsed_jsonl="outputs/parsed_6EQE.jsonl",
                         fixed_jsonl="outputs/fixed_6EQE.jsonl")
print("scorers built")

## Cell 6 — Equivalence check (GATE)

The one check that matters before scaling. If this raises, the `tied_featurize`
indices in `BatchScorer.__init__` need re-checking against this clone's
`protein_mpnn_utils.py` -- ProteinMPNN's return tuple is positional, so an
upgrade can reshuffle it silently.

In [ ]:
check_equivalence(batch, ref, target.wildtype_seq)

Time the speedup while you are here -- worth a line in the report.

In [ ]:
import time
t0 = time.time(); ref.score(target.wildtype_seq);        t_sub = time.time() - t0
t0 = time.time(); batch.score([target.wildtype_seq]*8);  t_bat = time.time() - t0
print(f"subprocess, 1 seq: {t_sub:.1f}s   batch, 8 seqs: {t_bat:.2f}s")

## Cell 7 — Small run

Confirm the loop before spending compute. What to look at, in order:

1. **`n_mut` per variant** -- must be <= `max_mutations` and mostly single digits.
   Anything in the hundreds means the constraint or the sampling is not taking effect.
2. **`known hotspot positions hit`** -- overlap with FAST-PETase / DuraPETase positions.
3. **`EXACT literature mutations`** -- same position *and* same amino acid. Rare, and
   the single most citable thing a run can produce.

In [ ]:
import time
from polyphage import (foldability_objective, novelty_objective, run,
                       report_front, print_front, log_run, summarize)

objectives = [foldability_objective(batch), novelty_objective(target)]
names = [o.name for o in objectives]

config = dict(pop_size=12, n_gen=5, max_mutations=15, seed=1)
t0 = time.time()
res = run(target, objectives, **config)
wall = time.time() - t0

reports = report_front(target, res, names)
print_front(reports)
print(summarize(reports))
log_run(config, reports, wall)

## Cell 8 — Plot the front

A front that is one point, or a dead-straight line, means the two objectives are
not actually in conflict -- fix that before adding more.

In [ ]:
from polyphage import plot_front

wt_objs = {
    "mpnn_nll": float(batch.score([target.wildtype_seq])[0]),
    "neg_n_mutations": 0.0,
}
fig, ax = plot_front(reports, wildtype_objectives=wt_objs)
fig.savefig("pareto_front.png", dpi=150)

## Cell 9 — Scale up

Only after Cells 7 and 8 look sane.

`max_mutations=10` on purpose: same regime as DuraPETase (10) and FAST-PETase (5),
so the comparison is like-for-like.

In [ ]:
config = dict(pop_size=40, n_gen=25, max_mutations=10, seed=1)
t0 = time.time()
res = run(target, objectives, **config)
wall = time.time() - t0

reports = report_front(target, res, names)
print_front(reports)
log_run(config, reports, wall)

## Cell 10 — Save the run log

Colab wipes the filesystem. Download this every session: the run-to-run
progression is the spine of the report and cannot be reconstructed later.

In [ ]:
from google.colab import files
files.download('runs.jsonl')
files.download('pareto_front.png')

---

# Part 2 — Is it real, and is it stable?

Two questions the run above cannot answer:

1. **Was T140D luck?** 15 known hotspot positions among 262 mutable ones means a
   9-mutation variant hits *some* hotspot about 40% of the time by chance.
   Cell 11 measures this against an explicit null instead of guessing.
2. **Does any of this survive heat?** The ProteinMPNN score says "this sequence
   fits this backbone". It says nothing about temperature. Cells 12-15 add a
   real stability objective.

## Cell 11 — Randomization control (the validity check)

Generates thousands of random variants with the same mutation count, and asks
how often chance alone reproduces what the search found.

Read the `p` values: `p < 0.05` means chance reproduces the result less than 5%
of the time. The exact-match p-value is the one that matters -- position-level
overlap is weak evidence on its own.

In [ ]:
from polyphage import run_control

control = run_control(target, reports, n_samples=5000)

## Cell 12 — Install ThermoMPNN

[ThermoMPNN](https://github.com/Kuhlman-Lab/ThermoMPNN) (Kuhlman Lab, MIT licence)
predicts **ddG** -- the change in folding free energy when you mutate a residue.
**Negative ddG = more stable.** It is built on ProteinMPNN's own embeddings, so
it shares the backbone already loaded here.

In [ ]:
%cd /content
!git clone https://github.com/Kuhlman-Lab/ThermoMPNN.git
!pip install -q pytorch-lightning omegaconf
%cd /content/ProteinMPNN
print("cloned")

## Cell 13 — Discover the interface (do not skip)

The ThermoMPNN README does not document `custom_inference.py`'s flags, and this
project has already lost time to a confidently-guessed identifier that did not
exist. Print the real help text and read it before running anything.

If the flags differ from `--pdb` / `--chain` / `--out`, pass the right ones
through `extra_args` in the next cell, or tell me and I will patch
`StabilityOracle.build`.

In [ ]:
!python /content/ThermoMPNN/custom_inference.py -h

## Cell 14 — Build the ddG table (once)

`custom_inference.py` runs *site-saturation mutagenesis*: every position against
every amino acid, in one pass. Cached as a table, so scoring a variant inside the
optimiser is just lookups -- no model calls in the loop.

In [ ]:
from polyphage import StabilityOracle

oracle = StabilityOracle(
    thermompnn_dir="/content/ThermoMPNN",
    pdb_path="inputs/design_target/6EQE.pdb",
    target=target,
    chain="A",
)
oracle.build()   # pass extra_args=[...] if Cell 13 showed different flags

print("\nMost stabilising single mutations ThermoMPNN predicts:")
for rn, aa, ddg in oracle.best_stabilising(15):
    wt = target.wildtype_seq[target.residue_numbers.index(rn)]
    print(f"  {wt}{rn}{aa}  ddG={ddg:+.3f}")

Worth a look before optimising: do any of those top single mutations coincide
with the FAST-PETase set (S121E, D186H, R224Q, N233K, R280A) or the DuraPETase
set (117, 119, 140, 159, 165, 168, 180, 188, 214, 280)? If ThermoMPNN already
ranks known-good mutations highly, say so in the report -- it means the search
starts from a well-informed prior.

In [ ]:
from polyphage import FAST_PETASE, DURA_PETASE

top = oracle.best_stabilising(100)
for rn, aa, ddg in top:
    tags = []
    if FAST_PETASE.get(rn) == aa: tags.append("FAST-PETase")
    if DURA_PETASE.get(rn) == aa: tags.append("DuraPETase")
    if tags:
        print(f"  {rn}{aa}  ddG={ddg:+.3f}   <-- {', '.join(tags)}")

## Cell 15 — Three-objective run

Now the search optimises:

- `mpnn_nll` -- does it fold (lower better)
- `ddg` -- does it survive heat (lower better, negative = stabilising)
- `neg_n_mutations` -- how far from wildtype (fewer mutations is easier to test)

These genuinely conflict, which is the point: stabilising mutations often cost
foldability, and both usually cost mutation count. The Pareto front is the map of
those trade-offs.

In [ ]:
import time
from polyphage import foldability_objective, novelty_objective, stability_objective
from polyphage import run, report_front, print_front, log_run

objectives = [
    foldability_objective(batch),
    stability_objective(oracle),
    novelty_objective(target),
]
names = [o.name for o in objectives]

config = dict(pop_size=40, n_gen=25, max_mutations=10, seed=1)
t0 = time.time()
res = run(target, objectives, **config)
wall = time.time() - t0

reports = report_front(target, res, names)
print_front(reports)
log_run({**config, "objectives": names}, reports, wall)

## Cell 16 — Control the new front, then repeat across seeds

One run recovering a known mutation could be luck in the search itself, separate
from luck in the null. Several seeds recovering it is a property of the method.

In [ ]:
control = run_control(target, reports, n_samples=5000)

In [ ]:
from polyphage import repeat_seeds_summary

per_seed = {}
for seed in [1, 2, 3, 4, 5]:
    r = run(target, objectives, pop_size=40, n_gen=25, max_mutations=10,
            seed=seed, verbose=False)
    per_seed[seed] = report_front(target, r, names)
    print(f"seed {seed}: {len(per_seed[seed])} variants")

summary = repeat_seeds_summary(per_seed)
print("\nrecovery across seeds:", summary["recovery_rate"])

---

## What this does and does not show

**Does:** produce variants that a structure model considers foldable and a
stability model considers more heat-tolerant than wildtype, and recover mutations
that were independently confirmed in the lab.

**Does not:** prove any of these degrade PET faster, or at all. Nothing here
measures catalysis. ddG predicts folding stability, not activity, and the two
often trade against each other -- rigidifying a protein can stiffen the active
site it needs for chemistry.

## What is still missing for the full goal

The remaining gap is *activity* -- does it actually cut plastic. That needs a
second stage, because it requires a 3D structure per candidate rather than a
sequence:

1. **Fold** the top candidates with ESMFold or ColabFold.
2. **Check the triad survived** -- Ser160 OG to His237 NE2 should stay near 3 A.
   A variant that scores well but distorts the triad is dead on arrival.
3. **Dock** a PET oligomer (MHET/BHET analog) with AutoDock Vina or GNINA, and
   measure the distance from Ser160's oxygen to the ester carbonyl carbon it has
   to attack.

This is a screen over the top ~20 candidates, not something that can run inside
the loop -- folding each candidate takes seconds to minutes. That is the next
build.